# Regresión Logística — Detección de Spam

---

## ¿Qué es la Regresión Logística?

La **Regresión Logística** es un algoritmo de clasificación que, a diferencia de la regresión lineal, predice **probabilidades** en lugar de valores continuos. Se usa cuando la variable de salida es una clase discreta (por ejemplo: spam / no spam).

La clave está en la **función sigmoide**, que transforma cualquier número real en un valor entre 0 y 1:

$$\sigma(z) = \frac{1}{1 + e^{-z}} \qquad \text{donde} \qquad z = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_n x_n$$

Esa salida se interpreta como **P(clase = 1 | x)**, es decir, la probabilidad de que el ejemplo pertenezca a la clase positiva.

La **regla de decisión** es simple:

$$\hat{y} = \begin{cases} 1 & \text{si } \sigma(z) \geq 0.5 \\ 0 & \text{si } \sigma(z) < 0.5 \end{cases}$$

---

## Problema a resolver

Clasificar correos electrónicos como **SPAM** o **HAM** (correo legítimo), usando el dataset TREC 2007 preprocesado.

- **Dataset**: [`processed_data.csv`](https://www.kaggle.com/datasets/imdeepmind/preprocessed-trec-2007-public-corpus-dataset)
- **Ruta**: `data/processed_data.csv` (relativa al notebook)

El flujo completo será:

| Paso | Descripción |
|------|-------------|
| 1 | Carga del dataset |
| 2 | Exploración inicial |
| 3 | Limpieza de datos (nulos, duplicados) |
| 4 | Preprocesamiento del texto (Bag of Words) |
| 5 | Entrenamiento del modelo |
| 6 | Evaluación (accuracy, matriz de confusión) |
| 7 | Análisis del impacto del tamaño del dataset |

---
## Paso 0 — Importar librerías

| Librería | Para qué la usamos |
|----------|--------------------|
| `pandas` | Cargar y manipular el dataset en forma de tabla (DataFrame) |
| `numpy` | Operaciones numéricas sobre arrays |
| `CountVectorizer` | Convertir texto a vectores numéricos (Bag of Words) |
| `LogisticRegression` | Entrenar el clasificador de spam |
| `accuracy_score` | Medir el porcentaje de aciertos del modelo |
| `confusion_matrix` | Ver los 4 tipos de error/acierto del clasificador |
| `matplotlib` | Crear gráficas |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

%matplotlib inline

---
## Paso 1 — Carga del Dataset

El CSV debe estar en `data/processed_data.csv` relativo al directorio donde se ejecuta este notebook.

El dataset contiene 5 columnas:

| Columna | Descripción |
|---------|-------------|
| `label` | **1 = SPAM**, **0 = HAM** (correo legítimo) |
| `subject` | Asunto del correo |
| `email_to` | Dirección de destino |
| `email_from` | Dirección de origen |
| `message` | Cuerpo del mensaje |

In [ ]:
df = pd.read_csv("data/processed_data.csv")

print(f"Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Columnas: {df.columns.tolist()}")

df.head()

---
## Paso 2 — Exploración del Dataset

Antes de limpiar o modelar, entendemos la distribución de las clases.

In [ ]:
conteo = df['label'].value_counts()
total  = len(df)

print("Distribución de clases (ANTES de limpiar):")
print(f"  SPAM (label=1): {conteo[1]:>7,}  ({conteo[1]/total*100:.1f}%)")
print(f"  HAM  (label=0): {conteo[0]:>7,}  ({conteo[0]/total*100:.1f}%)")
print(f"  Total          : {total:>7,}")
print()
print("⚠️  El dataset está desbalanceado: hay ~2× más SPAM que HAM.")
print("   Esto es típico en datasets de spam del mundo real.")

In [ ]:
# Ver ejemplos reales de cada clase para entender los datos
print("─" * 65)
print("EJEMPLOS DE SPAM (label=1):")
print("─" * 65)
for subj in df[df['label'] == 1]['subject'].dropna().head(4):
    print(f"  → {subj[:70]}")

print()
print("─" * 65)
print("EJEMPLOS DE HAM (label=0) — correo legítimo:")
print("─" * 65)
for subj in df[df['label'] == 0]['subject'].dropna().head(4):
    print(f"  → {subj[:70]}")

---
## Paso 3 — Limpieza de Datos

Un modelo de Machine Learning solo es tan bueno como sus datos. Antes de entrenar, debemos asegurarnos de que el dataset no contenga ruido ni inconsistencias.

Realizaremos tres verificaciones:
1. **Valores nulos** — filas con información faltante en columnas clave
2. **Duplicados** — correos idénticos que podrían sesgar el modelo
3. **Valores inválidos** — etiquetas fuera del rango esperado {0, 1}

In [ ]:
# ── 3.1 ANÁLISIS DE VALORES NULOS ──────────────────────────────────────────
print("VALORES NULOS POR COLUMNA:")
print("-" * 40)
nulos = df.isnull().sum()
for col in df.columns:
    pct = nulos[col] / len(df) * 100
    barra = "▓" * int(pct / 2)
    print(f"  {col:<20}: {nulos[col]:>5,}  ({pct:.1f}%) {barra}")

print()
print("Conclusión:")
print("  • 'subject' y 'message' tienen algunos nulos → los trataremos como cadena vacía.")
print("  • 'label' no tiene nulos → no hay correos sin clasificar. ✓")
print("  • 'email_to'/'email_from' no se usarán en el modelo.")

In [ ]:
# ── 3.2 ANÁLISIS DE DUPLICADOS ──────────────────────────────────────────────
# Consideramos duplicado si un correo tiene el mismo asunto, remitente,
# destinatario Y cuerpo — es decir, el mismo correo enviado varias veces.
n_duplicados = df.duplicated(subset=['subject', 'email_from', 'email_to', 'message']).sum()

print(f"Registros duplicados encontrados: {n_duplicados:,}")
print(f"  → Representan el {n_duplicados/len(df)*100:.1f}% del total.")
print()
print("¿Por qué son un problema?")
print("  Si un correo spam aparece 10 veces en train, el modelo aprende")
print("  a clasificarlo con demasiado 'énfasis', sesgando los θ hacia ese patrón.")

In [ ]:
# ── 3.3 VALORES INVÁLIDOS EN 'label' ────────────────────────────────────────
valores_unicos = sorted(df['label'].unique())
print(f"Valores únicos en 'label': {valores_unicos}")

invalidos = df[~df['label'].isin([0, 1])]
print(f"Registros con label inválido: {len(invalidos)}")
print("✓ Todos los labels son 0 o 1. No hay limpieza necesaria aquí.")

In [ ]:
# ── 3.4 APLICAR LIMPIEZA ────────────────────────────────────────────────────
filas_antes = len(df)

df = df.drop_duplicates(
    subset=['subject', 'email_from', 'email_to', 'message']
).reset_index(drop=True)

filas_despues = len(df)

print("RESUMEN DE LIMPIEZA:")
print(f"  Filas antes de limpiar : {filas_antes:>7,}")
print(f"  Duplicados eliminados  : {filas_antes - filas_despues:>7,}")
print(f"  Filas tras limpiar     : {filas_despues:>7,}")

# Verificar que no quedan duplicados
restantes = df.duplicated(subset=['subject','email_from','email_to','message']).sum()
print(f"  Duplicados restantes   : {restantes:>7,}  ✓")

# Distribución final de clases
conteo_limpio = df['label'].value_counts()
total_limpio = len(df)
print()
print("Distribución de clases (TRAS la limpieza):")
print(f"  SPAM (label=1): {conteo_limpio[1]:>7,}  ({conteo_limpio[1]/total_limpio*100:.1f}%)")
print(f"  HAM  (label=0): {conteo_limpio[0]:>7,}  ({conteo_limpio[0]/total_limpio*100:.1f}%)")
print(f"  Total          : {total_limpio:>7,}")

---
## Paso 4 — Preprocesamiento del Texto

Los algoritmos de Machine Learning trabajan con **números**, no con texto. Por eso necesitamos convertir cada correo en un vector numérico.

### Modelo Bag of Words (BoW)

La idea es simple:

1. Se construye un **vocabulario** con las palabras más frecuentes del dataset.
2. Cada correo se representa como un **vector** donde cada posición indica cuántas veces aparece esa palabra en el correo.

Ejemplo simplificado con vocabulario: `["viagra", "perl", "hello", "debian"]`:

| Correo | viagra | perl | hello | debian |
|--------|--------|------|-------|--------|
| "Buy cheap viagra now!" | 1 | 0 | 0 | 0 |
| "perl debian hello world" | 0 | 1 | 1 | 1 |

### `CountVectorizer` de scikit-learn

```python
CountVectorizer(max_features=10000)
```

- `max_features=10000` → limitamos el vocabulario a las **10.000 palabras más frecuentes** del corpus. Palabras muy raras (que aparecen 1-2 veces) aportan poco información y aumentan el coste computacional.

### División temporal (sin mezclar datos)

Usamos un **split temporal**: entrenamos con los primeros correos (los más antiguos) y evaluamos con los más recientes. Esto simula el escenario real: el modelo no puede ver correos del futuro durante el entrenamiento.

> ⚠️ **Importante**: `CountVectorizer` solo hace `.fit_transform()` sobre los datos de entrenamiento. Sobre el test usamos solo `.transform()`. Si hiciéramos `.fit()` con datos de test, el modelo tendría acceso a información que no debería conocer (***data leakage***).

In [ ]:
# Combinamos asunto + cuerpo en un solo texto por correo.
# Los nulos se sustituyen por cadena vacía para no perder filas.
df['texto'] = df['subject'].fillna('') + ' ' + df['message'].fillna('')

print("Ejemplo de texto combinado (primeros 200 caracteres de la fila 0):")
print(df['texto'].iloc[0][:200], '...')

In [ ]:
# División temporal: 80% train, 20% test
split = int(len(df) * 0.8)

X_train_raw = df['texto'][:split]   # correos más antiguos → entrenamiento
X_test_raw  = df['texto'][split:]   # correos más recientes → evaluación
y_train     = df['label'][:split]
y_test      = df['label'][split:]

print(f"Ejemplos de entrenamiento: {len(X_train_raw):>7,}  ({len(X_train_raw)/len(df)*100:.0f}%)")
print(f"Ejemplos de evaluación:   {len(X_test_raw):>7,}  ({len(X_test_raw)/len(df)*100:.0f}%)")

In [ ]:
# Vectorización: Bag of Words con las 10.000 palabras más frecuentes
vectorizer = CountVectorizer(max_features=10_000)

X_train = vectorizer.fit_transform(X_train_raw)  # aprende vocabulario Y transforma
X_test  = vectorizer.transform(X_test_raw)        # solo transforma (vocabulario ya fijado)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"  → {X_train.shape[0]:,} correos × {X_train.shape[1]:,} palabras del vocabulario")
print()
print(f"Dimensiones de X_test:  {X_test.shape}")
print()
print("La matriz es DISPERSA (sparse): la mayoría de celdas son 0")
print(f"porque cada correo solo contiene una pequeña fracción del vocabulario.")
print(f"Densidad estimada: {X_train.nnz / (X_train.shape[0]*X_train.shape[1]) * 100:.2f}%")

---
## Paso 5 — Entrenamiento del Modelo

### Regresión Logística con muchas variables

Con 10.000 palabras en el vocabulario, el modelo aprende **10.000 parámetros** $\theta_i$, uno por cada palabra. El modelo calcula:

$$P(\text{SPAM} \mid \text{correo}) = \sigma\left(\sum_{i=1}^{10000} \theta_i \cdot \text{freq}_i\right)$$

Donde $\text{freq}_i$ es el número de veces que aparece la palabra $i$ en el correo.

### Interpretación de los $\theta_i$

| Valor de $\theta_i$ | Significado |
|---------------------|-------------|
| $\theta_i \gg 0$ | La palabra $i$ es **fuertemente indicativa de SPAM** |
| $\theta_i \approx 0$ | La palabra $i$ es **neutra** (no discrimina bien) |
| $\theta_i \ll 0$ | La palabra $i$ es **fuertemente indicativa de HAM** |

> `max_iter=1000`: con 10.000 variables, el optimizador necesita más iteraciones para converger que en problemas más simples.

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

print(f"Modelo entrenado correctamente.")
print(f"  Parámetros θ aprendidos: {clf.coef_.shape[1]:,}")
print(f"  → Un θ por cada palabra del vocabulario de 10.000 palabras.")

In [ ]:
# Inspeccionar qué palabras el modelo asocia más a cada clase
nombres_features = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]

# Ordenar por θ: mayor = más SPAM, menor = más HAM
indices_spam = np.argsort(coefs)[::-1][:15]
indices_ham  = np.argsort(coefs)[:15]

print("PALABRAS MÁS INDICATIVAS DE SPAM (θ positivo alto):")
print(f"  {'Palabra':<22} {'θ':>8}")
print("  " + "-" * 32)
for i in indices_spam:
    print(f"  {nombres_features[i]:<22} {coefs[i]:>8.3f}")

print()
print("PALABRAS MÁS INDICATIVAS DE HAM (θ negativo alto):")
print(f"  {'Palabra':<22} {'θ':>8}")
print("  " + "-" * 32)
for i in indices_ham:
    print(f"  {nombres_features[i]:<22} {coefs[i]:>8.3f}")

print()
print("Nota: El dataset proviene de listas de correo técnicas (Debian, R, etc.),")
print("por eso palabras como 'perl', 'debian' o 'function' son muy indicativas de HAM.")

---
## Paso 6 — Predicción y Evaluación

### ¿Qué hace `predict`?

Para cada correo del conjunto de test:
1. Calcula $z = \theta^T x$ (combinación lineal de palabras y sus pesos)
2. Aplica la sigmoide: $p = \sigma(z)$
3. Si $p \geq 0.5$ → predice **SPAM** (label=1)
4. Si $p < 0.5$ → predice **HAM** (label=0)

### `predict_proba`

Devuelve la **probabilidad** de cada clase: `[P(HAM), P(SPAM)]` para cada correo.

In [ ]:
y_pred = clf.predict(X_test)
probs  = clf.predict_proba(X_test)  # columna 0 = P(HAM), columna 1 = P(SPAM)

etiqueta = lambda v: "SPAM" if v == 1 else "HAM"

print(f"{'Predicción':<12} {'P(HAM)':<10} {'P(SPAM)':<10} {'Real':<10} {'¿Correcto?'}")
print("-" * 57)
for i in range(8):
    pred = etiqueta(y_pred[i])
    real = etiqueta(list(y_test)[i])
    ok   = "✓" if pred == real else "✗ ERROR"
    print(f"{pred:<12} {probs[i][0]:<10.3f} {probs[i][1]:<10.3f} {real:<10} {ok}")

In [ ]:
acc = accuracy_score(y_test, y_pred)
n_test = len(y_test)
n_errores = int((1 - acc) * n_test)

print("═" * 42)
print("  MÉTRICAS DE EVALUACIÓN")
print("═" * 42)
print(f"  Accuracy:      {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Correos test:  {n_test:,}")
print(f"  Aciertos:      {n_test - n_errores:,}")
print(f"  Errores:       {n_errores:,}")
print("═" * 42)

### Matriz de Confusión

El accuracy por sí solo no cuenta toda la historia. La **matriz de confusión** desglosa los 4 tipos de resultado posibles:

```
                     Predicción
                  HAM    |   SPAM
Real HAM  → [Verdad. Neg.]  [Falso Pos.]   ← HAM marcado como SPAM
Real SPAM → [Falso Neg.]   [Verdad. Pos.]  ← SPAM marcado como HAM
```

En un filtro de spam:
- **Falsos Negativos** (SPAM que se cuela): molesto, pero tolerable
- **Falsos Positivos** (HAM bloqueado): ⚠️ más grave — el usuario puede perder emails importantes

Por eso, para un filtro de spam es preferible pecar de "permisivo" antes que bloquear correos legítimos.

In [ ]:
cm   = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['HAM (0)', 'SPAM (1)'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title("Matriz de Confusión\n", fontsize=13)
ax.set_xlabel("Predicción del modelo", fontsize=11)
ax.set_ylabel("Etiqueta real", fontsize=11)
plt.tight_layout()
plt.show()

# Extraer valores para el análisis
tn, fp, fn, tp = cm.ravel()
print(f"\nDesglose:")
print(f"  Verdaderos Negativos (HAM bien clasificado):     {tn:>6,}")
print(f"  Falsos Positivos     (HAM marcado como SPAM):   {fp:>6,}  ← emails legítimos bloqueados")
print(f"  Falsos Negativos     (SPAM que se cuela):       {fn:>6,}  ← spam no detectado")
print(f"  Verdaderos Positivos (SPAM bien detectado):     {tp:>6,}")
print()
print(f"Precision (SPAM): {tp/(tp+fp):.4f}  → de cada correo marcado como SPAM, {tp/(tp+fp)*100:.1f}% lo era realmente")
print(f"Recall    (SPAM): {tp/(tp+fn):.4f}  → el modelo detecta el {tp/(tp+fn)*100:.1f}% de los spams reales")
print()
print("Conclusión: el modelo bloquea pocos correos legítimos (bajo FP),")
print("aunque deja pasar algo de spam (FN). Comportamiento deseable en un filtro de email.")

---
## Paso 7 — Impacto del Tamaño del Dataset

Una pregunta clave en Machine Learning: **¿cuántos datos necesitamos para que el modelo funcione bien?**

Entrenamos el mismo modelo con subconjuntos de tamaño creciente y evaluamos en el **mismo conjunto de test** para comparar de forma justa.

In [ ]:
tamaños    = [500, 1000, 5000, 10000, 30000, 40000, 50000, split]
resultados = []

print(f"{'n (train)':<12} {'Accuracy':>10}")
print("-" * 24)

for n in tamaños:
    vec    = CountVectorizer(max_features=10_000)
    X_tr_n = vec.fit_transform(df['texto'][:n])
    X_te_n = vec.transform(X_test_raw)

    modelo = LogisticRegression(max_iter=1000)
    modelo.fit(X_tr_n, df['label'][:n])

    acc_n = accuracy_score(y_test, modelo.predict(X_te_n))
    resultados.append(acc_n)
    print(f"n={n:<9,} {acc_n:>10.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(tamaños, resultados, 'bo-', linewidth=2, markersize=7)
ax.axhline(y=resultados[-1], color='green', linestyle='--', alpha=0.5,
           label=f'Accuracy máximo = {resultados[-1]:.3f}')

for x, y_val in zip(tamaños, resultados):
    ax.annotate(f'{y_val:.3f}', (x, y_val),
                textcoords="offset points", xytext=(0, 10),
                ha='center', fontsize=8)

ax.set_xlabel("Número de correos de entrenamiento", fontsize=11)
ax.set_ylabel("Accuracy en test", fontsize=11)
ax.set_title("Curva de Aprendizaje: efecto del tamaño del dataset", fontsize=12)
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

**Conclusiones de la curva de aprendizaje:**

- El modelo aprende rápido: con solo **500 correos** ya supera el 90% de accuracy.
- El mayor salto ocurre entre 500 y 5.000 ejemplos — zona de mayor retorno de datos.
- Entre 30.000 y 50.000 hay una ligera meseta, pero el accuracy sigue creciendo.
- El modelo aún **no ha llegado a su techo** con este dataset — más datos probablemente seguirían mejorando el rendimiento.

---
## Resumen

| Paso | Descripción | Resultado |
|------|-------------|----------|
| 1 | Carga del dataset | 75.419 correos, 5 columnas |
| 2 | Exploración | 66.6% SPAM · 33.4% HAM (desbalanceado) |
| 3 | Limpieza | 1.513 duplicados eliminados → 73.906 correos limpios |
| 4 | Vectorización | Bag of Words · vocabulario de 10.000 palabras |
| 5 | Entrenamiento | LogisticRegression · 10.000 parámetros θ aprendidos |
| 6 | Evaluación | Accuracy ≈ 99% · bajos FP (pocos emails legítimos bloqueados) |
| 7 | Curva aprendizaje | El modelo mejora con más datos, aún sin llegar al techo |

### Próximos pasos para mejorar el modelo

- Usar **TF-IDF** en lugar de Bag of Words → penaliza palabras muy frecuentes que aportan poco
- Ajustar el **umbral de decisión** de 0.5 según el coste asimétrico (FP vs FN)
- Probar **Naive Bayes** → muy eficiente y eficaz en clasificación de texto
- Evaluar con métricas adicionales: F1-score, ROC-AUC, precision@recall